# Exploratory Data Analysis with SQL

This notebook reproduces the course SQL lab using a local SQLite table rather than requiring a hosted DB2 service. It uses the IBM Skills Network `Spacex.csv` dataset, which is a separate scraped table from the 90-row machine-learning dataset.

In [ ]:
import sqlite3, requests, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'; DATA.mkdir(exist_ok=True)
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv'
csv_path = DATA / 'Spacex.csv'
if not csv_path.exists():
    r = requests.get(url, timeout=45); r.raise_for_status(); csv_path.write_bytes(r.content)
spacex = pd.read_csv(csv_path)
print(spacex.shape, list(spacex.columns))
display(spacex.head())

In [ ]:
with sqlite3.connect(DATA / 'spacex_sql.sqlite') as conn:
    spacex.to_sql('SPACEXTBL', conn, if_exists='replace', index=False)
    queries = {
        'unique_launch_sites': "SELECT DISTINCT launch_site FROM SPACEXTBL ORDER BY launch_site",
        'nasa_crs_payload_mass': "SELECT SUM(payload_mass__kg_) AS total_payload_mass_kg FROM SPACEXTBL WHERE customer = 'NASA (CRS)'",
        'average_f9_v1_1_payload': "SELECT ROUND(AVG(payload_mass__kg_),0) AS avg_payload_mass_kg FROM SPACEXTBL WHERE booster_version LIKE 'F9 v1.1%'",
        'first_ground_pad_success': "SELECT MIN(DATE) AS first_date FROM SPACEXTBL WHERE landing__outcome = 'Success (ground pad)'",
        'success_vs_other': "SELECT CASE WHEN landing__outcome LIKE 'Success%' THEN 'Success' ELSE 'Other' END AS outcome_group, COUNT(*) AS rows FROM SPACEXTBL GROUP BY outcome_group",
        'landing_outcome_counts': "SELECT landing__outcome, COUNT(*) AS rows FROM SPACEXTBL GROUP BY landing__outcome ORDER BY rows DESC",
    }
    for title, sql in queries.items():
        print('\n', title)
        display(pd.read_sql_query(sql, conn))

## Interpretation

The published IBM lab reports four distinct launch-site strings, 45,596 kg total payload mass for NASA (CRS), a rounded 2,534 kg average for F9 v1.1 and the first successful ground-pad landing on 2015-12-22. The SQL table has 101 rows and is not the same dataset as the smaller model table. Re-run the queries to refresh values from the downloaded CSV.